In [2]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import warnings
warnings.filterwarnings("ignore")

In [3]:
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
model = joblib.load(
    "/content/drive/MyDrive/Internship project/best_message_classifier.pkl"
)

vectorizer = joblib.load(
    "/content/drive/MyDrive/Internship project/best_tfidf_vectorizer.pkl"
)

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [6]:
df = pd.read_csv(
    "/content/drive/MyDrive/Internship project/preprocessed_whatsapp_dataset.csv"
)

print(df.shape)

df.head()

(19825, 24)


,Person,Chat_Name,Timestamp,Sender,Message,Timestamp_Original,Date,Time,Year,Month,...,Minute,Is_Media,Is_Location,Is_Deleted,Is_Call,Is_URL,Character_Count,Word_Count,Is_Weekend,Is_Night
0,ARAVIND MENON,Chat wiith Coach Manoj,2026-01-04 07:00:00,Coach Manoj,"Morning! New year, new goals right? What are w...","04/01/26, 07:00 AM",2026-01-04,07:00:00,2026,1,...,0,False,False,False,False,False,132,22,True,False
1,ARAVIND MENON,Chat wiith Coach Manoj,2026-01-04 08:30:00,You,Honestly just consistency. Roster makes it har...,"04/01/26, 08:30 AM",2026-01-04,08:30:00,2026,1,...,30,False,False,False,False,False,116,20,True,False
2,ARAVIND MENON,Chat wiith Coach Manoj,2026-01-04 08:32:00,Coach Manoj,Fair enough. Let's do a flexible split then — ...,"04/01/26, 08:32 AM",2026-01-04,08:32:00,2026,1,...,32,False,False,False,False,False,155,30,True,False
3,ARAVIND MENON,Chat wiith Coach Manoj,2026-01-04 08:35:00,You,That works. When can we start?,"04/01/26, 08:35 AM",2026-01-04,08:35:00,2026,1,...,35,False,False,False,False,False,30,6,True,False
4,ARAVIND MENON,Chat wiith Coach Manoj,2026-01-04 08:36:00,Coach Manoj,Tomorrow morning if you're free. 6 AM slot open.,"04/01/26, 08:36 AM",2026-01-04,08:36:00,2026,1,...,36,False,False,False,False,False,48,9,True,False


In [7]:
df["Processed_Message"] = df["Message"].astype(str)

In [10]:
nltk.download("punkt_tab")

lemmatizer = WordNetLemmatizer()

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "]+",
    flags=re.UNICODE
)

def preprocess(text):

    text = text.lower()

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"<media omitted>", "", text, flags=re.IGNORECASE)

    text = re.sub(
        r"\d{1,2}/\d{1,2}/\d{2},?\s+\d{1,2}:\d{2}\s*(am|pm)?",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = emoji_pattern.sub("", text)

    text = re.sub(r"[^\w\s\u0D00-\u0D7F]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)


df["Processed_Message"] = df["Processed_Message"].apply(preprocess)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [11]:
X = vectorizer.transform(
    df["Processed_Message"]
)

In [12]:
df["Predicted_Risk"] = model.predict(X)

df[
    [
        "Message",
        "Predicted_Risk"
    ]
].head(20)

,Message,Predicted_Risk
0,"Morning! New year, new goals right? What are w...",Normal
1,Honestly just consistency. Roster makes it har...,Normal
2,Fair enough. Let's do a flexible split then — ...,Normal
3,That works. When can we start?,Normal
4,Tomorrow morning if you're free. 6 AM slot open.,Normal
5,6 AM works only if I'm not flying that day 😂 l...,Normal
6,"Confirmed, I'm off tomorrow. 6 AM works.",Normal
7,"Perfect, see you then. Come in comfortable clo...",Suspicious
8,Good first session. Your base fitness is decen...,Normal
9,Yeah I figured. Long duty hours don't leave mu...,Normal


In [13]:
risk_score = {
    "Normal":0,
    "Suspicious":1,
    "High Risk":2
}

df["Risk_Score"] = (
    df["Predicted_Risk"]
    .map(risk_score)
)

In [14]:
chat_summary = (
    df
    .groupby(
        ["Person","Chat_Name"]
    )
    .agg(
        Total_Messages=("Message","count"),
        Normal=("Predicted_Risk",lambda x:(x=="Normal").sum()),
        Suspicious=("Predicted_Risk",lambda x:(x=="Suspicious").sum()),
        High_Risk=("Predicted_Risk",lambda x:(x=="High Risk").sum()),
        Average_Risk=("Risk_Score","mean"),
        Max_Risk=("Risk_Score","max")
    )
    .reset_index()
)

chat_summary.head()

,Person,Chat_Name,Total_Messages,Normal,Suspicious,High_Risk,Average_Risk,Max_Risk
0,ARAVIND MENON,Chat wiith Coach Manoj,92,81,7,4,0.163043,2
1,ARAVIND MENON,Chat with Achanz,57,47,6,4,0.245614,2
2,ARAVIND MENON,Chat with Alan,63,52,9,2,0.206349,2
3,ARAVIND MENON,Chat with Ammaa,67,57,7,3,0.194030,2
4,ARAVIND MENON,Chat with Aparna,94,80,10,4,0.191489,2


In [15]:
def risk_level(avg, high):

    if high > 0:
        return "High"

    elif avg >= 0.5:
        return "Medium"

    else:
        return "Low"


chat_summary["Risk_Level"] = chat_summary.apply(
    lambda row: risk_level(
        row["Average_Risk"],
        row["High_Risk"]
    ),
    axis=1
)

In [16]:
person_summary = (
    chat_summary
    .groupby("Person")
    .agg(
        Total_Chats=("Chat_Name","count"),
        Total_Messages=("Total_Messages","sum"),
        High_Risk_Chats=("High_Risk",lambda x:(x>0).sum()),
        Average_Risk=("Average_Risk","mean")
    )
    .reset_index()
)

person_summary.head()

,Person,Total_Chats,Total_Messages,High_Risk_Chats,Average_Risk
0,ARAVIND MENON,22,1341,19,0.200196
1,Abdul Rahman chats,21,535,18,0.473443
2,Abhijith +2,40,973,17,0.143011
3,Alan techie,29,2681,11,0.063388
4,Arjun Toyota,47,1272,19,0.139867


In [17]:
chat_summary.to_csv(
    "/content/drive/MyDrive/Internship project/dashboard_chat_summary.csv",
    index=False
)

person_summary.to_csv(
    "/content/drive/MyDrive/Internship project/dashboard_person_summary.csv",
    index=False
)

df.to_csv(
    "/content/drive/MyDrive/Internship project/predicted_messages.csv",
    index=False
)

print("Dashboard files created successfully!")

Dashboard files created successfully!
